# 🎬 الحي 404 — إنتاج الأصوات المصرية مجانًا

ده دفتر تشغيل مباشر للحلقة الأولى، باستخدام نموذج **VoiceTut-TTS** مفتوح المصدر. لا يحتاج Runway أو ElevenLabs أو اشتراك مدفوع. استخدم Google Colab المجاني إذا توفر GPU (Runtime → Change runtime type → T4 GPU). **الـGPU المجاني غير مضمون**.

**أولًا:** شغّل خلية الإعداد، وبعدها خلية معاينة الأصوات، واسمع الفرق بين ليلى ووداد وصابر. **لا تشغّل اعتماد الأصوات إلا لو اتأكدت من اللهجة المصرية وطبيعة الصوت.** مفيش أي أصوات متولدة مسبقًا في الملفات دي.


In [ ]:
# Setup; Run this once. This installs the upstream public open-weight speech model.
import os, subprocess, sys
from pathlib import Path
if not Path("/content/404/.git").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/omarnet0109-jpg/404.git","/content/404"],check=True)
os.chdir("/content/404")
subprocess.run([sys.executable,"-m","pip","install","-q","git+https://github.com/k2-fsa/OmniVoice.git"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","voicetut-tts"],check=True)
try:
    import torch
    print("CUDA available:",torch.cuda.is_available())
except ImportError:
    print("PyTorch not detected; check GPU runtime.")


In [ ]:
# Free Egyptian previews: one sample per actor (Yasmin, Hanan, Shaker).
import subprocess, sys
from IPython.display import Audio, display, Markdown
subprocess.run([sys.executable,"production/free_tts.py","--preview"],check=True)
for person in ["layla","widad","saber"]:
    print("Voice:",person)
    display(Audio(filename=f"renders/episode-01/voice_previews/{person}.wav"))


## اعتماد الأصوات وتوليد الحوار

بعد الاستماع، لو ليلى ووداد وصابر أصواتهم مناسبة، شغّل الخلية التالية. لو اللهجة أو العمر مش مناسبين، غيّر أسماء الأصوات في `config/free_voices.json` قبل الاعتماد، ثم أعد المعاينات.

الخلية تنتج **9 ملفات WAV** مطابقة لجمل السيناريو الأصلية. هذا **صوت فقط**؛ تحريك الشخصيات ومزامنة الشفايف مرحلة منفصلة تحتاج محرّك فيديو فعلي.


In [ ]:
# SET APPROVE = True ONLY after you have listened to all three samples.
APPROVE = False
if not APPROVE:
    print("Preview is ready. Approve deliberately after listening, then rerun.")
else:
    import json
    from pathlib import Path
    cfg=Path("config/free_voices.json")
    data=json.loads(cfg.read_text(encoding="utf-8"))
    data["voices_previewed_and_approved"]=True
    cfg.write_text(json.dumps(data,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    subprocess.run([sys.executable,"production/free_tts.py","--episode"],check=True)
    import zipfile
    zpath=Path("/content/404_episode_1_egyptian_voices.zip")
    with zipfile.ZipFile(zpath,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(Path("renders/episode-01/voices").glob("*.wav")):
            z.write(f,arcname=f.name)
    from google.colab import files
    files.download(str(zpath))
